# Hugging Face Ecosystem

The **Hugging Face ecosystem** is the backbone of modern open-source AI. In this notebook we explore its four main components:

- **Hub** — repository of 500k+ models, datasets, and Spaces
- **Transformers** — unified API to load and run any model
- **Datasets** — fast, memory-efficient dataset loading and processing
- **Spaces** — deploy and share ML apps

## Setup

In [ ]:
!pip install transformers datasets sentence-transformers pillow -q

## Imports

In [ ]:
from transformers import pipeline
from huggingface_hub import model_info, list_models
import torch

device = 0 if torch.cuda.is_available() else -1
print(f"PyTorch {torch.__version__} | device: {'GPU' if device == 0 else 'CPU'}")

## 1. The Hugging Face Hub

The Hub at [huggingface.co](https://huggingface.co) hosts models, datasets, and Spaces. You can inspect any model programmatically.

In [ ]:
# Inspect a model card from the Hub
info = model_info("distilbert-base-uncased-finetuned-sst-2-english")

print(f"Model ID   : {info.modelId}")
print(f"Author     : {info.author}")
print(f"Downloads  : {info.downloads:,}")
print(f"Likes      : {info.likes}")
print(f"Pipeline   : {info.pipeline_tag}")
print(f"Library    : {info.library_name}")
print(f"\nTags: {info.tags[:8]}")

In [ ]:
# List top text-classification models by downloads
print("Top text-classification models on the Hub:\n")
models = list(list_models(filter="text-classification", sort="downloads", limit=5))
for m in models:
    print(f"  {m.downloads:>10,} downloads  {m.modelId}")

## 2. Transformers Library — 3 Tasks in 5 Lines Each

The `pipeline()` API abstracts all complexity. Here we show three different tasks with minimal code.

In [ ]:
# Task 1: Sentiment Analysis
sentiment = pipeline("sentiment-analysis", device=device)
print("=== Sentiment Analysis ===")
for text in ["Hugging Face is amazing!", "This model is too slow."]:
    r = sentiment(text)[0]
    print(f"  [{r['label']} {r['score']:.2f}] {text}")

# Task 2: Fill-Mask (BERT-style masked language modelling)
unmasker = pipeline("fill-mask", model="bert-base-uncased", device=device)
print("\n=== Fill-Mask ===")
for sentence in [
    "The capital of France is [MASK].",
    "Hugging Face is a company focused on [MASK] intelligence.",
]:
    top = unmasker(sentence)[0]
    print(f"  {sentence}")
    print(f"  → '{top['token_str']}'  (score {top['score']:.3f})\n")

# Task 3: Text Generation (GPT-2)
generator = pipeline("text-generation", model="gpt2", device=device)
print("=== Text Generation ===")
result = generator(
    "Open source AI is transforming the way we",
    max_new_tokens=30,
    do_sample=False,
)
print(" ", result[0]["generated_text"])

## 3. Model Caching

When you first run a model, weights are downloaded and cached locally. Subsequent runs are instant.

```
~/.cache/huggingface/hub/   ← default cache (or HF_HOME in Docker)
├── models--distilbert-base-uncased/
├── models--gpt2/
└── ...
```

You can pre-download models with `huggingface-cli download <model-id>`.

In [ ]:
import os
cache_dir = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface/hub"))
print(f"Model cache directory: {cache_dir}")

if os.path.exists(cache_dir):
    cached = [d for d in os.listdir(cache_dir) if d.startswith("models--")]
    print(f"Cached models ({len(cached)}):")
    for m in cached[:10]:
        print(f"  {m.replace('models--', '').replace('--', '/')}")
else:
    print("Cache directory not found — models will be downloaded on first use.")